<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Understand the Lagrangian

**Keep the clock exactly calibrated while finding settings with a small error score.**

The four observations and squared-error objective stay fixed. The calibration equality also stays fixed. The new idea is an auxiliary function called the Lagrangian: it helps find settings that respect that equality.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Keep the decision and the requirement separate

The recorded errors are $-2,-1,-1,0$ min at times $0,1,2,3$ h. Choose an initial correction $q$ in min and a rate correction $r$ in min/h. For reading $i=1,\ldots,4$, the remaining error is $y_i=d_i+q+rt_i$, where $d_i$ and $t_i$ are its recorded error and time. The score is $\phi(q,r)=\sum_{i=1}^{4}y_i^2$, in min².

Exact calibration at one hour requires $q+(1\,\mathrm h)r=1\,\mathrm{min}$. Using numerical values in these units, the **constraint residual** is

$$
h(q,r)=q+r-1=0.
$$

The unrestricted settings $(1.9,-0.6)$ give the small score 0.2 min², but $h=0.3$: calibration fails. The settings $(1.5,-0.5)$ give score 0.5 min² and satisfy calibration. A smaller score cannot excuse a failed requirement.

The code keeps remaining errors, the score, and feasibility separate. It also records the score's slopes with respect to the two settings.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Give the constraint a role in the calculation

The **Lagrangian** combines the objective and the constraint residual:

$$
L(q,r,\nu)=\phi(q,r)+\nu h(q,r).
$$

| Quantity | Role |
|:---|:---|
| $\phi(q,r)$ | Measures the clock's squared error |
| $h(q,r)$ | Reports whether exact calibration holds |
| $\nu$ | A multiplier solved together with the settings |
| $L(q,r,\nu)$ | An auxiliary function used to balance the slopes |

For every calibrated setting, $h=0$, so $L=\phi$ regardless of $\nu$. Away from calibration, the added term changes the function. This gives the constraint a role in its slopes, even though the original objective remains $\phi$.

The multiplier is not another clock setting or a preference weight chosen by the analyst. It may be positive, zero, or negative. Its units make $\nu h$ consistent with the score. We solve for a balance and still require $h=0$; we do not minimize $L$ freely over the settings and multiplier together.

In this comparison, changing $\nu$ from 0 to 2 leaves the physical errors unchanged. The feasible setting keeps $L=0.5$; the infeasible setting changes from $L=0.2$ to $L=0.8$ and remains rejected.


In [ ]:
def evaluate_lagrangian(decision, multiplier):
    result = evaluate_clock(decision)
    result["lagrangian"] = (result["objective"]
                            + multiplier * result["equality_residual"])
    return result


for decision in ([1.5, -0.5], [1.9, -0.6]):
    for multiplier in (0.0, 2.0):
        result = evaluate_lagrangian(decision, multiplier)
        print(f"Settings {decision}, nu = {multiplier:.0f}: "
              f"score = {result['objective']:.2f}, L = {result['lagrangian']:.2f}, "
              f"feasible = {result['feasible']}")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Balance slopes without leaving calibration

A slope tells us how the score changes when a setting changes. At a constrained minimum, the objective may still decrease in a direction that violates calibration. Its own slopes therefore need not be zero.

The multiplier allows the constraint's slopes to balance the objective's slopes. In derivative notation, the **Lagrange multiplier conditions** for this example are

$$
\frac{\partial L}{\partial q}=0,\qquad
\frac{\partial L}{\partial r}=0,\qquad h=0.
$$

These mean: balance both slopes and keep calibration. The clock score is smooth and the equality is a line with nonzero gradient, so these are valid necessary conditions here. They locate a candidate; they do not by themselves prove a minimum.

At $(q,r)=(1.5,-0.5)$, the objective slopes are $-2$ and $-2$. Since $h=q+r-1$, its slopes are 1 and 1. With $\nu=2$, the added slopes are $+2$ and $+2$, giving zero in both directions. These slopes use numerical values in the stated units.

The top row of the figure shows the unchanged score on the equality. The lower rows show why the added term still matters: it changes the slopes.
<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/03-2_constrained_optimization_lagrange_multipliers/assets/lagrangian_score_and_slopes.png" alt="At q equal to 1.5, r equal to minus 0.5, and multiplier 2, the equality residual is zero and the Lagrangian equals the objective value 0.5. For each setting, objective slope minus 2 plus constraint contribution 2 gives Lagrangian slope zero." width="512" style="display: block; max-width: 100%; height: auto; margin: 0; background-color: #fff;">
</div>


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

The code solves for $q$, $r$, and $\nu$ together. The first two rows represent the balanced slopes; the last row imposes calibration. Their coefficients come from the four squared errors. The result is $(q,r)=(1.5,-0.5)$ and $\nu=2$.


In [ ]:
# Unknowns are q, r, and the equality multiplier, in that order.
coefficients = np.array([
    [8.0, 12.0, 1.0],  # q slope plus the equality contribution
    [12.0, 28.0, 1.0], # r slope plus the equality contribution
    [1.0, 1.0, 0.0],   # exact calibration
])
right_hand_side = np.array([8.0, 6.0, 1.0])
correction, rate, equality_multiplier = np.linalg.solve(coefficients, right_hand_side)
candidate = evaluate_clock([correction, rate])
constraint_contribution = equality_multiplier * np.array([1.0, CALIBRATION_TIME])
print(f"q = {correction:.2f} min, r = {rate:.2f} min/h, nu = {equality_multiplier:.2f}")
print("Objective slopes:", candidate["gradient"])
print("Added constraint slopes:", constraint_contribution)
print("Lagrangian slopes (rounded):", np.round(candidate["gradient"] + constraint_contribution, 8) + 0.0)
print("Calibration holds:", candidate["feasible"])
print(f"Clock score: {candidate['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Read the result using nearby clock settings

Keep calibration by using $r=1-q$. The settings with $q=1.4$ and $q=1.6$ both give 0.56 min², while $q=1.5$ gives 0.50 min². Thus small feasible moves in either direction increase the score.

The short calculation below illustrates the result. For this quadratic example, one identity checks every calibrated setting:

$$
\phi(q,1-q)=0.5+6(q-1.5)^2.
$$

The added square is never negative and is zero only at $q=1.5$. This establishes the unique minimum on the equality, beyond the few settings printed by the code.


In [ ]:
for correction in (1.4, 1.5, 1.6):
    rate = REQUIRED_CORRECTION - correction
    result = evaluate_clock([correction, rate])
    print(f"q = {correction:.1f}, r = {rate:.1f}: "
          f"feasible = {result['feasible']}, score = {result['objective']:.2f} min²")
